In [1]:
import dgl.nn as dglnn
from dgl import from_networkx
import torch.nn as nn
import torch as th
import torch.nn.functional as F
import dgl.function as fn
from dgl.data.utils import load_graphs
import networkx as nx
import pandas as pd
import socket
import struct
import random
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

In [2]:
data = pd.read_csv('bot_iot.csv')

/tmp/ipykernel_74300/3198158916.py:1: DtypeWarning: Columns (7,9) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv('bot_iot.csv')


In [3]:
data.drop(columns=['subcategory','pkSeqID','stime','flgs','attack','state','proto','seq'],inplace=True)

In [4]:
data.rename(columns={"category": "label"},inplace = True)

In [5]:
data.label.value_counts()

label
DDoS              1926624
DoS               1650260
Reconnaissance      91082
Normal                477
Theft                  79
Name: count, dtype: int64

In [6]:
le = LabelEncoder()
le.fit_transform(data.label.values)
data['label'] = le.transform(data['label'])

In [7]:
data['saddr'] = data.saddr.apply(str)
data['sport'] = data.sport.apply(str)
data['daddr'] = data.daddr.apply(str)
data['dport'] = data.dport.apply(str)

In [8]:
data['saddr'] = data.saddr.apply(lambda x: socket.inet_ntoa(struct.pack('>I', random.randint(0xac100001, 0xac1f0001))))

In [9]:
data['saddr'] = data['saddr'] + ':' + data['sport']
data['daddr'] = data['daddr'] + ':' + data['dport']

In [10]:
data.drop(columns=['sport','dport'],inplace=True)
label_ground_truth = data[["saddr", "daddr","label"]]
data = pd.get_dummies(data, columns = ['flgs_number','state_number', 'proto_number'])

In [11]:
data = data.reset_index()
data.replace([np.inf, -np.inf], np.nan,inplace = True)

In [12]:
data.fillna(0,inplace = True)

In [13]:
label_ground_truth = data[["saddr", "daddr","label"]]

In [14]:
data.drop(columns=['index'],inplace=True)

In [15]:
data

,saddr,daddr,pkts,bytes,ltime,dur,mean,stddev,sum,min,...,state_number_7,state_number_8,state_number_9,state_number_10,state_number_11,proto_number_1,proto_number_2,proto_number_3,proto_number_4,proto_number_5
0,172.25.140.127:49960,192.168.100.7:80,8,1980,1.528089e+09,7.056393,0.068909,0.068909,0.137818,0.000000,...,False,False,False,False,False,True,False,False,False,False
1,172.21.209.161:-1,192.168.100.147:-1,2,120,1.528089e+09,0.000131,0.000131,0.000000,0.000131,0.000131,...,False,False,False,False,False,False,True,False,False,False
2,172.26.45.134:49962,192.168.100.7:80,8,2126,1.528089e+09,7.047852,0.064494,0.064494,0.128988,0.000000,...,False,False,False,False,False,True,False,False,False,False
3,172.25.142.211:49964,192.168.100.7:80,8,2024,1.528089e+09,7.047592,0.064189,0.064189,0.128378,0.000000,...,False,False,False,False,False,True,False,False,False,False
4,172.23.142.160:49966,192.168.100.7:80,8,2319,1.528089e+09,7.046841,0.063887,0.063887,0.127774,0.000000,...,False,False,False,False,False,True,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3668517,172.21.248.16:35064,192.168.100.3:22,6,434,1.529381e+09,0.013165,0.013165,0.000000,0.013165,0.013165,...,False,False,False,False,False,True,False,False,False,False
3668518,172.16.9.191:35066,192.168.100.3:22,6,434,1.529381e+09,0.000574,0.000574,0.000000,0.000574,0.000574,...,False,False,False,False,False,True,False,False,False,False
3668519,172.16.175.161:35070,192.168.100.3:22,31,5472,1.529381e+09,2.874302,2.874302,0.000000,2.874302,2.874302,...,False,False,False,False,False,True,False,False,False,False
3668520,172.28.181.114:43001,192.168.100.150:4433,2,134,1.529381e+09,0.000003,0.000003,0.000000,0.000003,0.000003,...,False,False,False,False,False,True,False,False,False,False


In [16]:
scaler = StandardScaler()
cols_to_norm = list(set(list(data.iloc[:, 2:].columns ))  - set(list(['label'])) )
data[cols_to_norm] = scaler.fit_transform(data[cols_to_norm])

In [17]:
X_train, X_test, y_train, y_test = train_test_split(
     data, label_ground_truth, test_size=0.3, random_state=42, stratify=label_ground_truth.label)

In [18]:
X_train['h'] = X_train[ cols_to_norm ].values.tolist()

In [19]:
#from dgl.data.utils import load_graphs   
#G = load_graphs("./data.bin")[0][0]

In [20]:
G = nx.from_pandas_edgelist(X_train, "saddr", "daddr", ['h','label'], create_using= nx.MultiGraph())

In [21]:
G = G.to_directed()

In [22]:
G = from_networkx(G,edge_attrs=['h','label'])

In [23]:
#from dgl.data.utils import save_graphs
#save_graphs("./data.bin", [G])


In [24]:
G.ndata['h'] = th.ones(G.num_nodes(), G.edata['h'].shape[1]) 

In [25]:
G.edata['train_mask'] = th.ones(len(G.edata['h']), dtype= th.bool)

In [26]:
#G = load_graphs("./bot_train_G.bin") [0][0]

In [27]:
# Eq1
G.ndata['h'] = th.ones(G.num_nodes(), G.edata['h'].shape[1])

In [28]:
G.edata['train_mask'] = th.ones(len(G.edata['h']), dtype=th.bool)

In [29]:
G.ndata['h'] = th.reshape(G.ndata['h'], (G.ndata['h'].shape[0], 1, G.ndata['h'].shape[1]))
G.edata['h'] = th.reshape(G.edata['h'], (G.edata['h'].shape[0], 1, G.edata['h'].shape[1]))

In [30]:
class MLPPredictor(nn.Module):
    def __init__(self, in_features, out_classes):
        super().__init__()
        self.W = nn.Linear(in_features * 2, out_classes)

    def apply_edges(self, edges):
        h_u = edges.src['h']
        h_v = edges.dst['h']
        score = self.W(th.cat([h_u, h_v], 1))
        return {'score': score}

    def forward(self, graph, h):
        with graph.local_scope():
            graph.ndata['h'] = h
            graph.apply_edges(self.apply_edges)
            return graph.edata['score']

In [31]:
G.ndata['h'].shape

torch.Size([2581017, 1, 55])

In [32]:
def compute_accuracy(pred, labels):
    return (pred.argmax(1) == labels).float().mean().item()

In [33]:
class SAGELayer(nn.Module):
    def __init__(self, ndim_in, edims, ndim_out, activation):
        super(SAGELayer, self).__init__()
        ### force to outut fix dimensions
        self.W_msg = nn.Linear(ndim_in + edims, ndim_out)
        ### apply weight
        self.W_apply = nn.Linear(ndim_in + ndim_out, ndim_out)
        self.activation = activation

    def message_func(self, edges):
        return {'m': self.W_msg(th.cat([edges.src['h'], edges.data['h']], 2))}

    def forward(self, g_dgl, nfeats, efeats):
        with g_dgl.local_scope():
            g = g_dgl
            g.ndata['h'] = nfeats
            g.edata['h'] = efeats
            # Eq4
            g.update_all(self.message_func, fn.mean('m', 'h_neigh'))
            # Eq5          
            g.ndata['h'] = F.relu(self.W_apply(th.cat([g.ndata['h'], g.ndata['h_neigh']], 2)))
            return g.ndata['h']


class SAGE(nn.Module):
    def __init__(self, ndim_in, ndim_out, edim, activation, dropout):
        super(SAGE, self).__init__()
        self.layers = nn.ModuleList()
        self.layers.append(SAGELayer(ndim_in, edim, 128, activation))
        self.layers.append(SAGELayer(128, edim, ndim_out, activation))
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, g, nfeats, efeats):
        for i, layer in enumerate(self.layers):
            if i != 0:
                nfeats = self.dropout(nfeats)
            nfeats = layer(g, nfeats, efeats)
        return nfeats.sum(1)

In [34]:
class Model(nn.Module):
    def __init__(self, ndim_in, ndim_out, edim, activation, dropout):
        super().__init__()
        self.gnn = SAGE(ndim_in, ndim_out, edim, activation, dropout)
        self.pred = MLPPredictor(ndim_out, 5)
    def forward(self, g, nfeats, efeats):
        h = self.gnn(g, nfeats, efeats)
        return self.pred(g, h)

In [36]:
from sklearn.utils import class_weight
class_weights = class_weight.compute_class_weight('balanced',
                                                 classes=np.unique(G.edata['label'].cpu().numpy()),
                                                 y=G.edata['label'].cpu().numpy())

In [37]:
class_weights = th.FloatTensor(class_weights)#.cuda()
criterion = nn.CrossEntropyLoss(weight = class_weights)

In [38]:
G = G#.to('cuda:0')

In [39]:
G.device

device(type='cpu')

In [40]:
G.ndata['h'].device  

device(type='cpu')

In [41]:
G.edata['h'].device  

device(type='cpu')

In [42]:
node_features = G.ndata['h']
edge_features = G.edata['h']

edge_label = G.edata['label']
train_mask = G.edata['train_mask']

model = Model(G.ndata['h'].shape[2], 128, G.ndata['h'].shape[2], F.relu, 0.2)#.cuda()

opt = th.optim.Adam(model.parameters())

# for epoch in range(1,14500):
#     pred = model(G, node_features,edge_features).cuda()
#     loss = criterion(pred[train_mask] ,edge_label[train_mask])
#     opt.zero_grad()
#     loss.backward()
#     opt.step()
#     if epoch % 100 == 0:
#         print('Epoch:', epoch ,' Training acc:', compute_accuracy(pred[train_mask], edge_label[train_mask]))

In [43]:
import os
os.remove("cluster_gcn.pkl")

In [44]:
import dgl
import dgl.nn as dglnn
import torch

In [45]:
num_parts = 1000
sampler = dgl.dataloading.ClusterGCNSampler(G, num_parts, prefetch_edata=["h", "label", "train_mask"]#, "val_mask", "test_mask"]
)

Convert a graph into a bidirected graph: 0.610 seconds, peak memory: 31.763 GB
Construct multi-constraint weights: 0.000 seconds, peak memory: 31.763 GB


[10:21:02] /opt/dgl/src/graph/transform/metis_partition_hetero.cc:89: Partition a graph with 2581017 nodes and 5135964 edges into 1000 parts and get 2497477 edge cuts


Metis partitioning: 26.206 seconds, peak memory: 32.186 GB


In [46]:
dataloader = dgl.dataloading.DataLoader(
    G, torch.arange(num_parts), sampler,
    batch_size=20, shuffle=True, drop_last=False, num_workers=4)

In [47]:
import time
for _ in range(10):
    t0 = time.time()
    history = model.train()
    for it, sg in enumerate(dataloader):
        x = sg.edata["h"]
        x2 = sg.ndata["h"]
        y = sg.edata["label"]
        m = sg.edata["train_mask"].bool()
        y_hat = model(sg, x2, x)
        loss = F.cross_entropy(y_hat[m], y[m])
        opt.zero_grad()
        loss.backward()
        opt.step()
        if it % 20 == 0:
            #acc = MF.accuracy(y_hat[m], y[m])
            #mem = torch.cuda.max_memory_allocated() / 1000000
            #print("Loss", loss.item(), "Acc", acc.item(), "GPU Mem", mem, "MB")
            print('Epoch:', it ,' Training acc:', compute_accuracy(y_hat[m], y[m]))
           # history = model.fit(y_hat[m], y[m], epochs=5, batch_size=1)
    tt = time.time()
    print(tt - t0)
    #durations.append(tt - t0)

/home/tselab/anaconda3/lib/python3.10/site-packages/dgl/dataloading/dataloader.py:1043: DGLWarning: Dataloader CPU affinity opt is not enabled, consider switching it on (see enable_cpu_affinity() or CPU best practices for DGL [https://docs.dgl.ai/tutorials/cpu/cpu_best_practises.html])
  dgl_warning(
/home/tselab/anaconda3/lib/python3.10/site-packages/dgl/backend/pytorch/tensor.py:449: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  assert input.numel() == input.storage().size(), (


Epoch: 0  Training acc: 0.7062339782714844
Epoch: 20  Training acc: 0.9834539890289307
Epoch: 40  Training acc: 0.9953739643096924
52.972941875457764
Epoch: 0  Training acc: 0.9972125291824341
Epoch: 20  Training acc: 0.9927954077720642
Epoch: 40  Training acc: 0.983660101890564
51.76565766334534
Epoch: 0  Training acc: 0.9659635424613953
Epoch: 20  Training acc: 1.0
Epoch: 40  Training acc: 0.9990800619125366
60.10446047782898
Epoch: 0  Training acc: 0.9950739145278931
Epoch: 20  Training acc: 0.9920454621315002
Epoch: 40  Training acc: 0.8074048161506653
59.63437342643738
Epoch: 0  Training acc: 0.8206947445869446
Epoch: 20  Training acc: 1.0
Epoch: 40  Training acc: 0.9862164258956909
49.95261836051941
Epoch: 0  Training acc: 0.9995574951171875
Epoch: 20  Training acc: 0.979468584060669
Epoch: 40  Training acc: 0.9966130256652832
49.368863105773926
Epoch: 0  Training acc: 0.9970831274986267
Epoch: 20  Training acc: 1.0
Epoch: 40  Training acc: 0.9961861371994019
50.571733713150024
E

In [48]:
model.eval()

Model(
  (gnn): SAGE(
    (layers): ModuleList(
      (0): SAGELayer(
        (W_msg): Linear(in_features=110, out_features=128, bias=True)
        (W_apply): Linear(in_features=183, out_features=128, bias=True)
      )
      (1): SAGELayer(
        (W_msg): Linear(in_features=183, out_features=128, bias=True)
        (W_apply): Linear(in_features=256, out_features=128, bias=True)
      )
    )
    (dropout): Dropout(p=0.2, inplace=False)
  )
  (pred): MLPPredictor(
    (W): Linear(in_features=256, out_features=5, bias=True)
  )
)

In [49]:
X_test['h'] = X_test[ cols_to_norm ].values.tolist()

In [50]:
#G_test = load_graphs("bot_test_G.bin") [0][0]
G_test = nx.from_pandas_edgelist(X_test, "saddr", "daddr", ['h','label'],create_using=nx.MultiGraph())
G_test = G_test.to_directed()
G_test = from_networkx(G_test,edge_attrs=['h','label'] )
#actual = G_test.edata.pop('label')
G_test.ndata['feature'] = th.ones(G_test.num_nodes(), 55)

In [51]:
G_test.ndata['feature'] = th.reshape(G_test.ndata['feature'], (G_test.ndata['feature'].shape[0], 1, G_test.ndata['feature'].shape[1]))

In [52]:
G_test.edata['h'] = th.reshape(G_test.edata['h'], (G_test.edata['h'].shape[0], 1, G_test.edata['h'].shape[1]))

In [58]:
G_test.edata['test_mask'] = th.ones(len(G_test.edata['h']), dtype=th.bool)

In [59]:
import os
os.remove("cluster_gcn.pkl")

In [60]:
num_parts = 2000
sampler = dgl.dataloading.ClusterGCNSampler(G_test, num_parts, prefetch_edata=["h", "label", "test_mask"]#, "val_mask", "test_mask"]
)

Convert a graph into a bidirected graph: 0.265 seconds, peak memory: 36.139 GB
Construct multi-constraint weights: 0.001 seconds, peak memory: 36.139 GB
Metis partitioning: 5.960 seconds, peak memory: 36.139 GB


[10:36:16] /opt/dgl/src/graph/transform/metis_partition_hetero.cc:89: Partition a graph with 1109928 nodes and 2201124 edges into 2000 parts and get 1072517 edge cuts


In [61]:
dataloader = dgl.dataloading.DataLoader(
    G_test, torch.arange(num_parts), sampler,
    batch_size=20, shuffle=True, drop_last=False, num_workers=4)

In [62]:
import torchmetrics.functional as MF

In [65]:
import timeit
totaltime=0
start_time = timeit.default_timer()
with torch.no_grad():
        val_preds, test_preds = [], []
        val_labels, test_labels = [], []
        for it, sg in enumerate(dataloader):
            x = sg.edata["h"]
            x2 = sg.ndata["feature"]
            y = sg.edata["label"]
            #m_val = sg.ndata["val_mask"].bool()
            m_test = sg.edata["test_mask"].bool()
            start_time = timeit.default_timer()
            y_hat = model(sg, x2, x)
            elapsed = timeit.default_timer() - start_time
            totaltime=totaltime+elapsed
           #val_preds.append(y_hat[m_val])
            #val_labels.append(y[m_val])
            test_preds.append(y_hat[m_test])
            test_labels.append(y[m_test])
            #break
       # val_preds = torch.cat(val_preds, 0)
       # val_labels = torch.cat(val_labels, 0)
        test_preds = torch.cat(test_preds, 0)
        test_labels = torch.cat(test_labels, 0)

        test_acc = MF.accuracy(
            test_preds,
            test_labels,
            task="multiclass",
           num_classes=5,
        )
        print("Test acc:", test_acc.item())
        #print("Test acc:", compute_accuracy(y_hat[m_test], y[m_test]))
        #tt = time.time()
        #print(tt - t0)ton_iot

Test acc: 0.9330592751502991


In [66]:
print(str(elapsed) + ' seconds')

0.301665947001311 seconds


In [67]:
print(str(totaltime) + ' seconds')

28.560461621993454 seconds


In [68]:
test_preds = test_preds.argmax(1)
test_preds = th.Tensor.cpu(test_preds).detach().numpy()

In [69]:
from sklearn.metrics import classification_report
#target_names = ['Normal', 'Abnormal']
print(classification_report(test_labels, test_preds, digits=4))#,target_names=target_names))

              precision    recall  f1-score   support

           0     0.8419    0.8253    0.8336     15658
           1     0.7921    0.8118    0.8018     12878
           2     0.9692    0.9333    0.9509       270
           3     0.9997    0.9998    0.9997     48752
           4     1.0000    0.8542    0.9213        48

    accuracy                         0.9331     77606
   macro avg     0.9206    0.8849    0.9015     77606
weighted avg     0.9333    0.9331    0.9331     77606



In [ ]:
#G_test = G_test.to('cuda:0')

In [ ]:
#th.cuda.empty_cache()

In [ ]:
#import timeit
#start_time = timeit.default_timer()
#node_features_test = G_test.ndata['feature']
#edge_features_test = G_test.edata['h']
#test_pred = model(G_test, node_features_test, edge_features_test).cuda()
#elapsed = timeit.default_timer() - start_time

In [ ]:
#print(str(elapsed) + ' seconds')

In [ ]:
test_pred = test_pred.argmax(1)

In [ ]:
test_pred = th.Tensor.cpu(test_pred).detach().numpy()

In [ ]:
edge_label = le.inverse_transform(actual)
test_pred = le.inverse_transform(test_pred)

In [ ]:
def plot_confusion_matrix(cm,
                          target_names,
                          title='Confusion matrix',
                          cmap=None,
                          normalize=True):
    
    import matplotlib.pyplot as plt
    import numpy as np
    import itertools

    accuracy = np.trace(cm) / float(np.sum(cm))
    misclass = 1 - accuracy

    if cmap is None:
        cmap = plt.get_cmap('Blues')

    plt.figure(figsize=(12, 12))
    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()

    if target_names is not None:
        tick_marks = np.arange(len(target_names))
        plt.xticks(tick_marks, target_names, rotation=45)
        plt.yticks(tick_marks, target_names)

    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]


    thresh = cm.max() / 1.5 if normalize else cm.max() / 2
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        if normalize:
            plt.text(j, i, "{:0.4f}".format(cm[i, j]),
                     horizontalalignment="center",
                     color="white" if cm[i, j] > thresh else "black")
        else:
            plt.text(j, i, "{:,}".format(cm[i, j]),
                     horizontalalignment="center",
                     color="white" if cm[i, j] > thresh else "black")


    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label\naccuracy={:0.4f}; misclass={:0.4f}'.format(accuracy, misclass))
    plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix

plot_confusion_matrix(cm = confusion_matrix(edge_label, test_pred), 
                      normalize    = False,
                      target_names = np.unique(edge_label),
                      title        = "Confusion Matrix")